# Configuración del entorno en Colab

Ejecutar al inicio de **cada sesión** de Colab. Hace lo siguiente:

1. Verifica la GPU asignada.
2. Monta Google Drive.
3. Clona el repositorio (o lo actualiza si ya existe).
4. Instala las dependencias.
5. Copia el dataset de Drive al disco local de Colab (`/content`) y lo descomprime en `data/raw/`.

> Se entrena leyendo desde el disco local, no desde Drive: leer miles de imágenes directo desde Drive es muy lento y deja a la GPU esperando.

**Estructura esperada en Drive** (ajustar `DRIVE_DATA_DIR` si es otra):
```
MyDrive/redes-parcial-p3/
  *.zip    imágenes de CCT-20 y/o anotaciones comprimidas
  *.json   anotaciones ECCV 2018 (si no están dentro de un zip)
```

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar / actualizar el repositorio

In [ ]:
import os

REPO_URL = 'https://github.com/Roxana-Vargas/redes-parcial-p3.git'
REPO_DIR = '/content/redes-parcial-p3'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 4. Dependencias

In [ ]:
!pip install -q -r requirements.txt

import ultralytics
ultralytics.checks()

## 5. Copiar el dataset de Drive a `/content`

In [ ]:
import glob, shutil, zipfile

DRIVE_DATA_DIR = '/content/drive/MyDrive/redes-parcial-p3'
RAW_DIR = os.path.join(REPO_DIR, 'data', 'raw')
os.makedirs(os.path.join(RAW_DIR, 'annotations'), exist_ok=True)

assert os.path.isdir(DRIVE_DATA_DIR), f'No existe {DRIVE_DATA_DIR}: revisar la ruta en Drive'

for zip_path in sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, '*.zip'))):
    print('Descomprimiendo', os.path.basename(zip_path), '...')
    local_zip = os.path.join('/content', os.path.basename(zip_path))
    shutil.copy(zip_path, local_zip)  # copiar primero: descomprimir desde Drive es lento
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(RAW_DIR)
    os.remove(local_zip)

for json_path in glob.glob(os.path.join(DRIVE_DATA_DIR, '*.json')):
    shutil.copy(json_path, os.path.join(RAW_DIR, 'annotations'))

!du -sh {RAW_DIR}/*

## 6. Verificación rápida

In [ ]:
n_images = sum(1 for p in glob.glob(os.path.join(RAW_DIR, '**', '*.jpg'), recursive=True))
json_files = glob.glob(os.path.join(RAW_DIR, '**', '*.json'), recursive=True)

print(f'Imágenes .jpg: {n_images:,}')
print('Anotaciones JSON:')
for p in sorted(json_files):
    print('  ', os.path.relpath(p, RAW_DIR))